In [ ]:
## Overview

This notebook implements the **offline data construction pipeline** for a GDPR-focused semantic retrieval Q&A system.

The objective is to build a **high-reliability, traceable knowledge base** for legal question answering, where all candidate answers are pre-validated before being served to end users.

Unlike typical LLM-based QA systems, this design **intentionally restricts the role of large language models** to an offline validation layer, rather than real-time answer generation.

This ensures:
- Deterministic and auditable outputs
- Reduced hallucination risk
- Suitability for compliance-sensitive use cases

---

### Pipeline Overview

The workflow consists of:

- Structuring GDPR legal text (Article 1–11) into normalized formats
- Applying **structure-aware semantic chunking** (Article → Paragraph → Subparagraph)
- Generating embeddings using a multilingual sentence transformer
- Constructing and validating QA pairs via an offline evaluation pipeline
- Using LLMs (ChatGPT API) **only for QA validation**, not for answer generation

---

### Design Principle

A key design principle is:

> **LLMs should not be the source of truth in legal QA systems.**

Instead, they are used only to **evaluate and validate structured knowledge**, ensuring that:

- Final answers remain consistent and traceable
- Legal hallucination risks are minimized
- The system remains suitable for high-stakes compliance scenarios

---

### Notes

- Outputs are removed for readability and security
- API keys are managed via environment variables and are not included


In [ ]:
## API Key Setup

This notebook requires an OpenAI API key.

Before running, please set your API key as an environment variable:

```python
import os
os.environ["OPENAI_API_KEY"] = "your-api-key"

In [ ]:
import pandas as pd

df=pd.read_csv("gdpr_raw.csv")
df_subset=df[df["article_no"].between(1,11)].copy()
df_subset.reset_index(drop=True,inplace=True)

# Normalize and clean GDPR text for downstream processing
# - Standardize text format
# - Remove formatting artifacts (newline, tab, extra spaces)
# - Merge EN/ZH content to support bilingual semantic retrieval

def clean_text(text):
  text=str(text)
  text=text.replace("\n"," ")
  text=text.replace("\t"," ")
  text=text.replace(" "," ")
  return text.strip()

# Combine English and Chinese into a unified representation
df_subset["clean_text"]=df_subset.apply(
  lambda row:clean_text(row["text_en"])+"\n"+clean_text(row["text_ch"]),
  axis=1
)
# Design Note:
# Combining EN/ZH text into a single representation ensures that
# semantic embeddings capture cross-lingual alignment,
# which is critical for bilingual retrieval scenarios.



# Baseline chunking strategy: fixed-length segmentation
# This serves as an initial reference implementation
def chunk_text(text,max_length=500):
  chunks=[]
  start=0
  while start<len(text):
    chunks.append(text[start:start+max_length])
    start+=max_length
  return chunks
# Design Note:
# Fixed-length chunking is used as a baseline for comparison.
# However, it does not respect legal document structure,
# which may lead to semantic fragmentation across chunks.


all_chunks=[]

for _,row in df_subset.iterrows():
  chunks=chunk_text(row["clean_text"],max_length=500)
  for i, chunk in enumerate(chunks):
    all_chunks.append({
      "article_no":row["article_no"],
      "paragraph_no":row["paragraph_no"],
      "subparagraph_no":row["subparagraph_no"],
      "chunk_id":i,
      "text":chunk
    })

chunk_df=pd.DataFrame(all_chunks)
chunk_df.head(10)

chunk_df.to_csv("gdpr_article_1_11_chunks.csv",index=False,encoding="utf-8-sig")
print("Chunked CSV has been generated.")

In [ ]:
import pandas as pd
chunk_df=pd.read_csv("gdpr_article_1_11_chunks.csv")
texts=chunk_df["text"].tolist()

print(len(texts))
print(texts[0][:100])

In [ ]:
# Generate semantic embeddings for each chunk
# These embeddings enable similarity-based retrieval in the QA phase

from sentence_transformers import SentenceTransformer
embedding_backend="sentence_transformers"

if embedding_backend=="sentence_transformers":
  embed_model=SentenceTransformer("all-MiniLM-L6-v2")

def embed_texts(texts):
  return embed_model.encode(texts,show_progress_bar=True)

embeddings=embed_texts(texts)
print(len(embeddings),len(embeddings[0]))

# Design Note:
# A lightweight embedding model is used here for efficiency.
# Higher-quality multilingual models will be introduced later
# after validating chunking quality.

In [ ]:
import pandas as pd
import numpy as np

CSV_PATH="gdpr_article_1_11_chunks.csv"
OUTPUT_PATH="gdpr_article_1_11_chunks_clean.csv"
MIN_LENGTH=100

def has_chinese(text):
  return any('\u4e00'<=c<='\u9fff' for c in text)

def has_english(text):
  return any(c.isalpha() and c.lower()<='z' for c in text)

def clean_subpara(val):
  if pd.isna(val):
    return ""
  return str(val).strip()

chunk_df=pd.read_csv(CSV_PATH,encoding='utf-8-sig')
print("Number of raw records:",len(chunk_df))
display(chunk_df.head())

In [ ]:
# Identify low-quality chunks that may degrade retrieval performance
# Criteria:
# - Too short (insufficient semantic content)
# - Missing bilingual information
# - Structurally inconsistent

problematic=[]

for idx,row in chunk_df.iterrows():
  text=str(row["text"]).strip()
  if (
    len(text)<MIN_LENGTH
    or not has_chinese(text)
    or not has_english(text)
  ):
    problematic.append({
      "index":idx,
      "article_no":row["article_no"],
      "paragraph_no":row["paragraph_no"],
      "subparagraph_no":row["subparagraph_no"],
      "chunk_id":row["chunk_id"],
      "length":len(text),
      "text_preview":text[:80]
    })

print(f"\nDetected {len(problematic)} potentially problematic chunks.")
if problematic:
  display(pd.DataFrame(problematic))

# Design Note:
# In legal retrieval systems, chunk quality directly affects answer reliability.
# This validation step helps detect low-information or structurally incomplete chunks
# that can introduce noise and reduce ranking precision
# before they propagate into the QA pipeline.

In [ ]:
# Merge semantically incomplete chunks while preserving legal structure
# Conditions:
# - Chunk is too short
# - Missing bilingual content
# - Within the same Article / Paragraph / Subparagraph

# Design Note:
# Instead of naive merging, we enforce structural constraints
# to preserve legal hierarchy, ensuring that merged chunks
# remain semantically coherent and legally valid.

In [ ]:
new_chunks=[]
skip_next=False

for i in range(len(chunk_df)):
  if skip_next:
    skip_next=False
    continue

  row=chunk_df.iloc[i]
  text=str(row["text"]).strip()

  need_merge=(
    len(text)<MIN_LENGTH
    or not has_chinese(text)
    or not has_english(text)
  )

  if need_merge and i+1<len(chunk_df):
    next_row=chunk_df.iloc[i+1]
    merged_text=text+" "+str(next_row["text"]).strip()
    new_chunks.append({
      "article_no":row["article_no"],
      "paragraph_no":row["paragraph_no"],
      "subparagraph_no":clean_subpara(row["subparagraph_no"]),
      "chunk_id":row["chunk_id"],
      "text":merged_text.strip()
    })
    skip_next=True
  else:
    new_chunks.append({
      "article_no":row["article_no"],
      "paragraph_no":row["paragraph_no"],
      "subparagraph_no":clean_subpara(row["subparagraph_no"]),
      "chunk_id":row["chunk_id"],
      "text":text
    })

clean_df=pd.DataFrame(new_chunks)
print("\nNumber of chunks after merging:",len(clean_df))
display(clean_df.head())

clean_df.to_csv(OUTPUT_PATH,index=False,encoding='utf-8-sig')
print(f"\nCleaned chunk file exported: {OUTPUT_PATH}")

In [ ]:
import pandas as pd

MIN_LENGTH=100
new_chunks=[]

i=0
while i<len(chunk_df):
  row=chunk_df.iloc[i]
  text=str(row["text"]).strip()
  current_chunk=row.to_dict()
  if len(text)<MIN_LENGTH and i+1<len(chunk_df):
    next_row=chunk_df.iloc[i+1]

    same_structure=(
      row["article_no"]==next_row["article_no"] and
      row["paragraph_no"]==next_row["paragraph_no"] and
      row["subparagraph_no"]==next_row["subparagraph_no"]
    )

    if same_structure:
      current_chunk["text"]=text+" "+str(next_row["text"]).strip()
      i+=1
  new_chunks.append(current_chunk)
  i+=1

fixed_df=pd.DataFrame(new_chunks)
fixed_df.to_csv("gdpr_article_1_11_chunks_fixed.csv",index=False,encoding="utf-8-sig")

print("Exported: gdpr_article_1_11_chunks_fixed.csv")
print("Number of chunks after merging:",len(fixed_df))

In [ ]:
import pandas as pd

df=pd.read_csv("gdpr_raw.csv")

df["text_en"]=df["text_en"].fillna("").astype(str).str.strip()
df["text_ch"]=df["text_ch"].fillna("").astype(str).str.strip()
df["citation"]=df["citation"].fillna("").astype(str).str.strip()
df["subparagraph_no"]=df["subparagraph_no"].apply(
  lambda x:str(x).strip().replace("(","").replace(")","")
  if pd.notna(x) else x
)

# Apply structure-aware chunking based on legal hierarchy:
# Article → Paragraph → Subparagraph
# This preserves legal semantics and improves retrieval accuracy

chunks=[]

for _,row in df.iterrows():
  article=str(int(row["article_no"])) if pd.notna(row["article_no"]) else ""
  para=str(int(row["paragraph_no"])) if pd.notna(row["paragraph_no"]) else ""
  sub=row["subparagraph_no"]

  # Generate structured chunk ID
  chunk_id=f"Art{article}_Para{para}"
  if pd.notna(sub):
    chunk_id+=f"_{sub}"

  # Construct bilingual chunk text (EN + ZH)
  # Keeping both languages together ensures semantic alignment
  chunk_text=(
    f"Article {article},Paragraph {para}"
    + (f" ({sub})" if pd.notna(sub) else "")
    + "\n\n"
    + "[EN]\n"
    + row["text_en"]
    + "\n\n"
    + "[ZH]\n"
    + row["text_ch"]
  )

  chunks.append({
    "chunk_id":chunk_id,
    "article_no":article,
    "paragraph_no":para,
    "subparagraph_no":sub,
    "content":chunk_text,
    "citation":row["citation"]
  })
final_df=pd.DataFrame(chunks)

OUTPUT_PATH="gdpr_structured_chunks_final.csv"
final_df.to_csv(OUTPUT_PATH,index=False,encoding="utf-8-sig")

# Design Note:
# The finalized structured chunk dataset serves as the canonical retrieval layer
# for subsequent embedding construction and QA validation.
print("Final structured chunks have been exported.")
print("Number of chunks:",len(final_df))

In [ ]:
# Data quality checks for chunk integrity

# 1. Check chunk length distribution
#    - detect overly short or excessively long chunks
#    - ensure chunks are suitable for embedding

# 2. Validate uniqueness of chunk IDs
#    - prevent duplication in retrieval index

# 3. Validate structural uniqueness
#    - ensure (Article, Paragraph, Subparagraph) combinations are not duplicated

In [ ]:
import pandas as pd

df=pd.read_csv("gdpr_structured_chunks_final.csv")

df["char_count"]=df["content"].astype(str).apply(len)

print("Chunk length statistics")
print(df["char_count"].describe())

too_short=df[df["char_count"]<80]
too_long=df[df["char_count"]>2000]

print("\nChunks shorter than 80 characters:",len(too_short))
print("\nChunks longer than 2000 characters:",len(too_long))

In [ ]:
dupes=df[df.duplicated("chunk_id",keep=False)]
print("Number of duplicate chunk IDs:",len(dupes))

struct_dupes=df[df.duplicated(
  ["article_no","paragraph_no","subparagraph_no"],
  keep=False
)]
print("Number of duplicated structural entries:",len(struct_dupes))

In [ ]:
missing_en=df[df["content"].str.contains(r"\[EN\\s*\n\s*\[ZH\]",regex=True)]
print("Chunks missing English content:",len(missing_en))

missing_zh=df[df["content"].str.endswith("[ZH]\n")]
print("Chunks missing Chinese content:",len(missing_zh))

In [ ]:
print("Citation field summary")
print(df["citation"].value_counts().head(10))

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

chunk_df=pd.read_csv("gdpr_structured_chunks_final.csv")
texts=chunk_df["content"].tolist()

print("Number of loaded chunks:",len(texts))

In [ ]:
# Generate semantic embeddings for GDPR chunks
# to support similarity-based retrieval in downstream QA testing

embed_model=SentenceTransformer("all-MiniLM-L6-v2")
embeddings=embed_model.encode(texts,show_progress_bar=True)

print("Embedding shape:",embeddings.shape)

In [ ]:
# Define a basic top-k semantic retrieval function
# Used to evaluate whether chunk embeddings capture relevant legal semantics

# Design Note:
# Retrieval is evaluated prior to answer generation to verify that
# the chunking strategy preserves sufficient legal semantics.
# This step acts as a validation layer before integrating LLM-based QA.

def retrieve_top_k(query,texts,embeddings,k=10):
  query_emb=embed_model.encode([query])
  sims=cosine_similarity(query_emb,embeddings)[0]
  top_k_idx=np.argsort(sims)[-k:][::-1]
  return [(chunk_df.iloc[i]["article_no"],
       chunk_df.iloc[i]["paragraph_no"],
       chunk_df.iloc[i]["subparagraph_no"],
       texts[i],
       sims[i]) for i in top_k_idx]

In [ ]:
# Initial Chinese query test
# Used to evaluate whether semantically relevant GDPR chunks can be retrieved before downstream QA refinement

query_ch="合法處理個人資料應符合哪些要件?"
results_ch=retrieve_top_k(query_ch,texts,embeddings,k=10)

print("\n=== Chinese QA Retrieval Results ===")
for i,(article,para,sub,text,score) in enumerate(results_ch,1):
  print(f"\nTop {i} (score={score:.4f})")
  print(f"Article {article}, Paragraph {para}, Subparagraph {sub}")
  print(text[:900])

In [ ]:
# Initial English query test
# Used to compare bilingual retrieval behavior and identify possible semantic fragmentation issues

query_en="What are the requirements for lawful processing of personal data?"
results_en=retrieve_top_k(query_en,texts,embeddings,k=10)

print("\n=== English QA Retrieval Results ===")
for i, (article,para,sub,text,score) in enumerate(results_en,1):
  print(f"\nTop {i} (score={score:.4f})")
  print(f"Article {article}, Paragraph {para}, Subparagraph {sub}")
  print(text[:900])

In [ ]:
import pandas as pd

# Load the finalized chunk dataset produced after structure-aware preprocessing
chunk_df=pd.read_csv("gdpr_structured_chunks_final.csv")
texts=chunk_df["content"].tolist()

In [ ]:
# Use a multilingual embedding model to better support bilingual legal text retrieval
# especially for Chinese-English GDPR chunk representations

# Design Note:
# A multilingual model is used because the corpus contains bilingual GDPR content (EN/ZH),
# and retrieval quality depends on preserving cross-lingual semantic alignment

# =========================
# Embedding & Retrieval Setup
# =========================

# Generate semantic embeddings for GDPR chunks
# These embeddings serve as the foundation for similarity-based retrieval in QA tasks
from sentence_transformers import SentenceTransformer
embed_model=SentenceTransformer("intfloat/multilingual-e5-base")

def embed_texts(texts):
  return embed_model.encode(texts,show_progress_bar=True,normalize_embeddings=True)

embeddings=embed_texts(texts)

print(len(embeddings),len(embeddings[0]))


In [ ]:
# ==============================
# Basic Retrieval Function
# ==============================

# Retrieve top-k most relevant chunks using cosine similarity
# Operates at subparagraph-level granularity for fine-grained semantic matching

# Design Note:
# Subparagraph-level retrieval maximizes recall of precise legal conditions,
# but may introduce fragmented context, which will be addressed later via aggregation.
# Retrieval quality is evaluated before introducing any answer-generation layer.
# This isolates the impact of chunking and embedding design on semantic alignment.

import numpy as np
def retrieve_top_k(query,texts,embeddings,k=10):

  # Encode query into embedding space
  query_emb=embed_model.encode([query])

  # Compute similarity scores
  sims=cosine_similarity(query_emb,embeddings)[0]

  # Select top-k results
  top_k_idx=np.argsort(sims)[-k:][::-1]
  return [(chunk_df.iloc[i]["article_no"],
       chunk_df.iloc[i]["paragraph_no"],
       chunk_df.iloc[i]["subparagraph_no"],
       texts[i],
       sims[i]) for i in top_k_idx]


In [ ]:
# ==============================
# Paragraph-level Aggregation
# ==============================

# Aggregate subparagraph-level retrieval results into paragraph-level outputs
# This improves coherence and reduces fragmentation in legal answer presentation

# Design Note:
# While fine-grained retrieval improves recall, legal interpretation often requires
# paragraph-level context. Aggregation bridges precision and interpretability.

from collections import defaultdict

def qa_with_paragraph_aggregation(query,chunk_df,texts,embeddings,k=20):

  # Step 1: retrieve fine-grained results
  results=retrieve_top_k(query,texts,embeddings,k=k)

  # Step 2: group results by (Article, Paragraph)
  grouped=defaultdict(list)

  for article,para,sub,text,score in results:
    key=(article,para)
    grouped[key].append({
      "subparagraph":sub,
      "text":text,
      "score":score
    })

  print(f"\n Query:{query}\n")

  # Step 3: display aggregated results
  for (article,para), items in grouped.items():
    print("="*60)
    print(f"Article{article},Paragraph{para}")
    print("-"*60)

    for item in sorted(items,key=lambda x:x["score"],reverse=True):
      sub=item["subparagraph"]
      score=item["score"]
      text=item["text"]

      print(f"\n Subparagraph {sub} (score={score:.4f})")
      print(text[:800])


In [ ]:
qa_with_paragraph_aggregation(
  query="合法處理個人資料應符合哪些要件?",
  chunk_df=chunk_df,
  texts=texts,
  embeddings=embeddings,
  k=20
)

In [ ]:
qa_with_paragraph_aggregation(
  query="What are the requirements for lawful processing of personal data?",
  chunk_df=chunk_df,
  texts=texts,
  embeddings=embeddings,
  k=20
)

In [ ]:
# ==============================
# Rule-based Query Augmentation (Baseline)
# ==============================

# Expand user queries with domain-relevant GDPR terminology
# to improve recall in semantic retrieval

# Design Note:
# Users often phrase questions differently from legal texts.
# Query augmentation helps align natural language queries with formal GDPR terminology.


# Generate expanded query variants using GDPR-related keywords
def expand_query_for_gdpr(query):
  expansions=[
    query,
    query+"GDPR principles",
    query+"lawfullness fairness transparency",
    query+"purpose limitation data minimization accuracy",
  ]
  return expansions

def qa_with_query_augmentation(query,chunk_df,texts,embeddings,k=20,debug=False):
  expanded_queries=expand_query_for_gdpr(query)

  all_scores=[]

  for q in expanded_queries:
    q_emb=embed_model.encode([q])
    sims=cosine_similarity(q_emb,embeddings)[0]
    all_scores.append(sims)

  # Optional debugging output to inspect the retrieval impact of each expanded query
  if debug:
    print("=== Query Augmentation Impact ===")
    for q,sims in zip(expanded_queries,all_scores):
      print(f"{q}->maxsimilarity:{np.max(sims):.4f}")

  # Combine similarity scores across expanded queries using max pooling
  # Each chunk retains its strongest semantic match
  final_scores=np.max(all_scores,axis=0)
  top_k_idx=np.argsort(final_scores)[-k:][::-1]

  results=[
    (
      chunk_df.iloc[i]["article_no"],
      chunk_df.iloc[i]["paragraph_no"],
      chunk_df.iloc[i]["subparagraph_no"],
      texts[i],
      final_scores[i]
    )
    for i in top_k_idx
  ]

  return results


In [ ]:
# Baseline retrieval setting (without paragraph-level aggregation)
# Used to compare fragmented outputs against aggregated results

# Design Note:
# This comparison highlights the trade-off between retrieval precision
# and answer-level coherence.

results=qa_with_query_augmentation(
  query="合法處理個人資料應符合哪些要件?",
  chunk_df=chunk_df,
  texts=texts,
  embeddings=embeddings,
  k=20,
  debug=True
)

for article,para,sub,text,score in results:
  print(f"Article{article},Paragraph{para},Subparagraph{sub},score={score:.4f}")

In [ ]:
# Utility function for displaying aggregated retrieval results

from collections import defaultdict

def display_with_paragraph_aggregation(query,retrieval_results):
  grouped=defaultdict(list)

  for article,para,sub,text,score in retrieval_results:
    key=(article,para)
    grouped[key].append({
      "subparagraph":sub,
      "text":text,
      "score":score
    })

  print(f"\n Query:{query}\n")

  for (article,para),items in grouped.items():
    print("="*70)
    print(f"Article{article},Paragraph{para}")
    print("-"*70)

    for item in sorted(items,key=lambda x:x["score"],reverse=True):
      sub=item["subparagraph"]
      score=item['score']
      text=item["text"]

      print(f"\n Subparagraph{sub} (score={score:.4f})")
      print(text[:800])

In [ ]:
query="合法處理個人資料應符合哪些要件?"

results=qa_with_query_augmentation(
  query=query,
  chunk_df=chunk_df,
  texts=texts,
  embeddings=embeddings,
  k=20,
)

display_with_paragraph_aggregation(query,results)

In [ ]:
query="What are the requirements for lawful processing of personal data?"

results=qa_with_query_augmentation(
  query=query,
  chunk_df=chunk_df,
  texts=texts,
  embeddings=embeddings,
  k=20,
)

display_with_paragraph_aggregation(query,results)

In [ ]:
# ==============================
# Enhanced English Query Augmentation
# ==============================

# Extend English queries with more explicit legal terminology
# to improve retrieval coverage for GDPR provisions

# Design Note:
# English queries often under-specify legal terminology.
# Additional expansion helps reduce semantic mismatch with statutory language.

def expand_query_for_gdpr_en_precise(query):
  return[
    query,
    query+"GDPR principles",
    query+"lawfullness fairness transparency",
    query+"purpose limitation data minimization accuracy",
    query+"storage limitation retention period data retention"
  ]

# Add additional GDPR-specific legal phrases for improved recall
def qa_with_query_augmentation_en_precise(query,chunk_df,texts,embeddings,k=20,debug=True):
  expanded_queries=expand_query_for_gdpr_en_precise(query)
  all_scores=[]

  for q in expanded_queries:
    q_emb=embed_model.encode([q])
    sims=cosine_similarity(q_emb,embeddings)[0]
    all_scores.append(sims)

  # Use max pooling across augmented query scores
  # to retain the strongest relevance signal for each chunk
  final_scores=np.max(all_scores,axis=0)

  if debug:
    print("=== Query Augmentation Impact ===")
    for q,sims in zip(expanded_queries,all_scores):
      print(f"{q}->max similarity:{np.max(sims):.4f}")

  top_k_idx=np.argsort(final_scores)[-k:][::-1]

  results=[
    (
      chunk_df.iloc[i]["article_no"],
      chunk_df.iloc[i]["paragraph_no"],
      chunk_df.iloc[i]["subparagraph_no"],
      texts[i],
      final_scores[i]
    )
    for i in top_k_idx
  ]

  return results

In [ ]:
# Utility function for displaying aggregated retrieval results

from collections import defaultdict

def display_with_paragraph_aggregation(query,retrieval_results):
  grouped=defaultdict(list)

  for article,para,sub,text,score in retrieval_results:
    key=(article,para)
    grouped[key].append({
      "subparagraph":sub,
      "text":text,
      "score":score
    })

  print(f"\n Query:{query}\n")

  for (article,para),items in grouped.items():
    print("="*70)
    print(f"Article{article},Paragraph{para}")
    print("-"*70)

    for item in sorted(items,key=lambda x:x["score"],reverse=True):
      sub=item["subparagraph"]
      score=item['score']
      text=item["text"]

      print(f"\n Subparagraph{sub}(score={score:.4f})")
      print(text[:800])

In [ ]:
# ==============================
# Retrieval Evaluation (Augmented Queries)
# ==============================

# Evaluate whether query augmentation improves retrieval robustness

# Design Note:
# Comparing baseline and enhanced query strategies helps isolate whether
# performance gains come from better query formulation or improved aggregation.

query_en="What are the requirements for lawful processing of personal data?"

results_en_precise=qa_with_query_augmentation_en_precise(
  query=query_en,
  chunk_df=chunk_df,
  texts=texts,
  embeddings=embeddings,
  k=20,
  debug=True
)

display_with_paragraph_aggregation(query_en,results_en_precise)

In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# ==============================
# Rule-based Multilingual Query Augmentation
# ==============================

# Expand queries based on detected legal intent (e.g., consent, retention, rights)
# Supports both English and Chinese inputs

# Design Note:
# This rule-based layer introduces lightweight intent recognition
# without requiring a full NLP classification model,
# improving recall in domain-specific retrieval tasks.
def expand_query_for_gdpr(query):
  expansions=[query]

  q=query.lower()

  # Retention / storage related queries
  if"保存" in q or "retention" in q:
    expansions+=[
      query+"storage limitation retention period data retention",
      query+"GDPR Article 5(1)(e)"
    ]

  # Consent / lawful basis
  if "同意" in q or "consent" in q:
    expansions+=[
      query+"consent lawful basis GDPR Article 6 7"
    ]

  # Data subject rights
  if "權利" in q or "rights" in q:
    expansions+=[
      query+"data subject rights GDPR"
    ]

  # Special categories of data
  if "特殊" in q or "special" in q:
    expansions+=[
      query+"special categories of personal data GDPR Article 9",
      query+"genetic biometric health data"
    ]

  # Territorial scope
  if "適用" in q or "scope" in q or "territorial" in q:
    expansions+=[
      query+"GDPR territorial scope material scope Article 3"
    ]

  # Security
  if "安全" in q or "security" in q:
    expansions+=[
      query+"personal data security GDPR Article 32",
    ]

  # General principles
  expansions+=[
    query+"GDPR principles lawfulness fairness transparency"
  ]

  return expansions

# ==============================
# Retrieval with Query Augmentation
# ==============================

# Perform retrieval using multiple expanded queries
# Combine results via max pooling

# Design Note:
# This approach improves recall by capturing multiple semantic interpretations
# of the same user query while maintaining ranking stability.
def qa_with_query_augmentation(query,chunk_df,texts,embeddings,k=20,debug=True):
  expanded_queries=expand_query_for_gdpr(query)
  all_scores=[]

  for q in expanded_queries:
    q_emb=embed_model.encode([q])
    sims=cosine_similarity(q_emb,embeddings)[0]
    all_scores.append(sims)

  if debug:
    print("=== query augmentation impact ===")
    for q,sims in zip(expanded_queries,all_scores):
      print(f"{q}->max similarity:{np.max(sims):.4f}")

  # Max pooling across query variants
  final_scores=np.max(all_scores,axis=0)
  top_k_idx=np.argsort(final_scores)[-k:][::-1]

  results=[
    (
      chunk_df.iloc[i]["article_no"],
      chunk_df.iloc[i]["paragraph_no"],
      chunk_df.iloc[i]["subparagraph_no"],
      texts[i],
      final_scores[i]
    ) for i in top_k_idx
  ]
  return results


In [ ]:
# Utility function for displaying aggregated retrieval results

from collections import defaultdict

def display_with_paragraph_aggregation(query,retrieval_results):
  grouped=defaultdict(list)
  for article, para,sub,text,score in retrieval_results:
    key=(article,para)
    grouped[key].append({
      "subparagraph":sub,
      "text":text,
      "score":score
    })

  print(f"\n Query:{query}\n")
  for (article,para),items in grouped.items():
    print("="*70)
    print(f"Article{article},Paragraph{para}")
    print("-"*70)
    for item in sorted(items,key=lambda x:x["score"],reverse=True):
      sub=item["subparagraph"]
      score=item["score"]
      text=item["text"]
      print(f"\n Subparagraph{sub}(score={score:.4f})")
      print(text[:800])


In [ ]:
# Sample Chinese test queries (for bilingual retrieval evaluation)

test_queries_zh=[
  "企業是否能處理特殊特殊類型之個人資料?",
  "資料主體有哪些權利?",
  "個人資料應保存多久?",
  "處理個人資料前是否需要取得同意?",
  "企業於歐盟提供商品或服務所涉之個人資料處理是否適用GDPR?",
  "企業應如何保護個人資料安全?"
]

for q in test_queries_zh:
  results=qa_with_query_augmentation(q,chunk_df,texts,embeddings,k=20,debug=True)
  display_with_paragraph_aggregation(q,results)


In [ ]:
# Sample English test queries

test_queries_en=[
  "Can companies process special categories of personal data?",
  "What rights do the data subject have?",
  "How long should personal data be retained?",
  "Is consent required before processing personal data?",
  "Is the processing of personal data involving the provision of products or services by companies subject to GDPR?",
  "How should companies protect the security of personal data?"
]

for q in test_queries_en:
  results=qa_with_query_augmentation(q,chunk_df,texts,embeddings,k=20,debug=True)
  display_with_paragraph_aggregation(q,results)


In [ ]:
# ==============================
# OpenAI API Setup
# ==============================

# Load API credentials from environment variables
# to avoid exposing sensitive keys in the notebook

import os

api_key = os.getenv("OPENAI_API_KEY")


In [ ]:
# Install and initialize OpenAI SDK
# Used only in the offline validation stage

!pip install --upgrade openai

from openai import OpenAI
client = OpenAI()


In [ ]:
# ==============================
# API Connectivity Check
# ==============================

# Run a minimal API call to verify that the environment is correctly configured
# before executing the offline QA pipeline

messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Hello, can you confirm your API works?"}
]

try:

    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=messages,
        max_tokens=50
    )


    print("=== ChatGPT Response ===")

    print(response.choices[0].message.content)

    print("\n=== Token Usage ===")
    print(f"Prompt tokens: {response.usage.prompt_tokens}")
    print(f"Completion tokens: {response.usage.completion_tokens}")
    print(f"Total tokens: {response.usage.total_tokens}")

except Exception as e:
    print("API call failed:", e)


In [ ]:
import os
import openai

openai.api_key = os.getenv("OPENAI_API_KEY")

if openai.api_key is None:
    raise ValueError("API key not set. Please configure OPENAI_API_KEY in environment variables.")
else:
    print("API key successfully loaded.")


In [ ]:
# Legacy answer-drafting function (retained for reference only)
# The newer version introduces stricter prompting and bilingual output control

def ask_chatgpt(question, context_texts, model="gpt-3.5-turbo", max_tokens=500, temperature=0.2):
    """
    question: user query
    context_texts: list of retrieved GDPR legal passages
    """

    context = "\n\n".join(context_texts)

    messages = [
        {"role": "system", "content": "You are a GDPR expert. Answer the question based strictly on the provided legal context. Respond in Chinese."},
        {"role": "user", "content": f"Answer the following question based on the provided legal context:\n{context}\n\nQuestion: {question}"}
    ]


    response = client.chat.completions.create(
        model=model,
        messages=messages,
        max_tokens=max_tokens,
        temperature=temperature
    )

    answer = response.choices[0].message.content
    return answer

In [ ]:
qa_results = {
    "企業是否能處理特殊特殊類型之個人資料?": [
        "Article9, Paragraph2.0 (部分法條，缺少(d)(f)(h)(j))...",
        "Article9, Paragraph3.0...",
        "Article9, Paragraph4.0..."
    ],
    "資料主體有哪些權利?": [
        "Article7, Paragraph3.0...",
        "Article6, Paragraph1.0...",
        "Article6, Paragraph4.0...",
        "Article7, Paragraph1.0...",
        "Article7, Paragraph4.0...",
        "Article9, Paragraph2.0(c)(e)...",
        "Article6, Paragraph3.0...",
        "Article4, Paragraph11.0...",
        "Article8, Paragraph1.0..."
    ],
    "個人資料應保存多久?": [
        "Article5, Paragraph1(e)...",
        "Article5, Paragraph1.0...",
        "Article6, Paragraph1.0...",
        "Article4, Paragraph3.0...",
        "Article6, Paragraph4.0...",
        "Article6, Paragraph3.0...",
        "Article10, Paragraph..."
    ],
    "處理個人資料前是否需要取得同意?": [
        "Article6, Paragraph1.0...",
        "Article7, Paragraph3.0...",
        "Article6, Paragraph4.0...",
        "Article7, Paragraph1.0...",
        "Article7, Paragraph4.0...",
        "Article5, Paragraph1.0...",
        "Article9, Paragraph2.0(c)(e)...",
        "Article6, Paragraph3.0...",
        "Article4, Paragraph11.0...",
        "Article8, Paragraph1.0..."
    ],
    "企業於歐盟提供商品或服務所涉之個人資料處理是否適用GDPR?": [
        "Article3, Paragraph1.0...",
        "Article3, Paragraph2.0...",
        "Article3, Paragraph3.0..."
    ],
    "企業應如何保護個人資料安全?": [
        "Article32, Paragraph1.0...",
        "Article32, Paragraph2.0...",
        "Article32, Paragraph3.0..."
    ]
}


for question, chunks in qa_results.items():
    print(f"=== Question: {question} ===\n")
    answer = ask_chatgpt(question, chunks)
    print(answer)
    print("\n" + "="*80 + "\n")


In [ ]:
!pip install --upgrade openai --quiet

In [ ]:
import os
import openai
api_key = os.getenv("OPENAI_API_KEY")

In [ ]:
# Utility function for safely printing multilingual output
# Used to preserve Chinese and English answer readability during offline QA review

def safe_print(*args, **kwargs):
    """Ensure proper rendering of multilingual or special characters"""
    print(*args, **kwargs, flush=True)

In [ ]:
# =========================
# Offline QA Drafting with LLM
# =========================

# Use the OpenAI API to draft structured answers from retrieved GDPR context chunks.
# This function is used only in the offline QA pipeline for validation and answer drafting,
# not in the final online LINE Bot response flow.

# Design Note:
# The LLM is used strictly for offline drafting and validation.
# Final user-facing answers are retrieved from a reviewed QA dataset,
# rather than generated in real time.

def ask_chatgpt(question, context_chunks, model="gpt-3.5-turbo", max_tokens=2000, language="zh"):
    """
    Draft a structured answer using retrieved GDPR context chunks.

    Parameters:
    - question: user-facing legal question
    - context_chunks: retrieved GDPR chunks used as supporting context
    - model: OpenAI model used in offline validation
    - max_tokens: maximum response length
    - language: output language ("zh" or "en")

    This function is part of the offline QA validation pipeline.
    It is used to generate and inspect answer drafts before finalizing the QA dataset.
    """
    context_text = "\n".join(context_chunks)

    # Design Note:
    # The model is explicitly constrained to operate within the provided legal context
    # to reduce hallucination and improve answer controllability.

    if language == "zh":
        system_msg = (
            "你是 GDPR 法律助理，請嚴格依照以下規則回答：\n"
            "1. 僅能依據提供的 context 條文內容回答，不得引用或推測未提供的條文。\n"
            "2. 回答時必須遵循 GDPR 的法律結構，並可清楚說明條文之間的邏輯關係（例如：原則與例外）。\n"
            "3. 若條文包含列舉式條件（paragraph / subparagraph），必須完整逐一描述所有 subparagraph 的實質內容，包括角色（如資料主體、企業）及行為。\n"
            "4. 不得使用模糊或開放式用語（例如：「包括但不限於」、「可能包含」）。\n"
            "5. 若 context 不足以支持完整法律結論，須明確說明哪些結論可以得出，哪些無法判斷，原因為何。\n"
            "6. 不得自行補充法律背景、立法目的或未出現在 context 中的解釋。\n"
            "7. 請使用清楚的條列式結構，並以法律答題風格作答。\n"
            "8. 必須保留 context 中每一個 subparagraph，並完整翻譯成中文，不可省略任何一點。"
        )
    else:
        system_msg = (
            "You are a GDPR legal assistant. Follow these rules strictly:\n"
            "1. Answer strictly based on the provided context only. Do not cite or assume provisions not included in the context.\n"
            "2. Follow the legal structure of the GDPR provisions and clearly explain their logical relationship (e.g., general rule and exceptions).\n"
            "3. If a provision contains enumerated paragraphs or subparagraphs, explicitly describe the substance of each item, including the roles (e.g., data subject, company) and actions.\n"
            "4. Do not use vague or open-ended language (e.g., 'including but not limited to', 'may include').\n"
            "5. If the context does not allow a complete legal conclusion, clearly distinguish what can be concluded and what cannot, and explain why.\n"
            "6. Do not add legal explanations, assumptions, or background information that are not present in the context.\n"
            "7. Use clear bullet points and structured legal-answer formatting.\n"
            "8. All enumerated subparagraphs must be fully listed and described. No item may be omitted."
        )

    messages = [
        {"role": "system", "content": system_msg},
        {"role": "user", "content": f"Question: {question}\n\nContext:\n{context_text}"}
    ]

    try:
        response = openai.chat.completions.create(
            model=model,
            messages=messages,
            temperature=0.0,
            max_tokens=max_tokens
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        return f"API call failed: {e}"


# =========================
# Retrieved Context Sets for Offline QA Validation
# =========================

# Each question is paired with a curated set of retrieved GDPR chunks.
# These context sets are used to draft and validate candidate answers
# before constructing the finalized QA knowledge base.

qa_results = {
    "企業是否能處理特殊類型之個人資料?": [
        "Article9, Paragraph1 (General prohibition on processing special categories of personal data)...",
        "Article9, Paragraph2(a) Explicit consent - 獲得資料主體明確同意",
        "Article9, Paragraph2(b) Employment, social security and social protection law - 就業、社會保障和社會保護法律下的處理",
        "Article9, Paragraph2(c) Vital interests - 為保護資料主體或其他自然人的生命利益而處理",
        "Article9, Paragraph2(d) Non-profit bodies - 非營利機構在非營利目的下，僅針對其成員或前成員的處理",
        "Article9, Paragraph2(e) Data manifestly made public by the data subject - 資料已由資料主體自行公開",
        "Article9, Paragraph2(f) Legal claims - 用於法律索賠或法院司法行為",
        "Article9, Paragraph2(g) Substantial public interest - 基於歐盟或成員國法律具有重大公共利益的處理",
        "Article9, Paragraph2(h) Health or social care - 用於預防、職業醫學、健康或社會護理目的",
        "Article9, Paragraph2(i) Public health - 用於公共衛生目的",
        "Article9, Paragraph2(j) Archiving, research and statistics - 用於檔案、研究和統計目的"
    ],
    "資料主體有哪些權利?": [
        "Article1-11 GDPR 僅規定本規則之目的、適用範圍、定義、資料處理原則及監管架構。",
        "Provided context does NOT include Articles 12-22, where specific data subject rights are enumerated.",
        "Therefore, no specific data subject rights can be identified or listed based solely on the provided Articles 1-11."
    ],
    "個人資料應保存多久?": [
        "Article5, Paragraph1(e) - 原則: 個人資料不得保存超過處理目的所需時間 (Personal data shall not be kept longer than necessary for processing purposes).",
        "Article5, Paragraph1(e) - 例外: 若用於公共利益、科學或歷史研究、統計目的，可延長保存，但須落實技術與組織措施保障資料主體權利 (May be stored longer for archiving in public interest, scientific/historical research, or statistical purposes, subject to technical and organizational safeguards)."
    ],
    "處理個人資料前是否需要取得同意?": [

        """Article 6, Paragraph 1 — Lawfulness of processing:
        Processing of personal data shall be lawful only if at least one of the following applies, including consent, performance of a contract, compliance with a legal obligation, protection of vital interests, performance of a task carried out in the public interest or in the exercise of official authority, or legitimate interests pursued by the controller.
        → This provision establishes that consent is one possible lawful basis for processing, but not a mandatory requirement in all cases.""",

        """Article 7, Paragraphs 1, 3 and 4 — Conditions for consent:
        Where processing is based on consent, the controller shall be able to demonstrate that the data subject has consented; the data subject has the right to withdraw consent at any time; withdrawal of consent shall not affect the lawfulness of processing based on consent before its withdrawal.
        → These provisions clarify the legal consequences and limits of relying on consent as a lawful basis, including the right of withdrawal.""",

        """Article 9, Paragraph 1 and Paragraph 2(a) — Special categories of personal data:
        Processing of special categories of personal data is in principle prohibited, unless the data subject has given explicit consent for one or more specified purposes.
        → This establishes a general prohibition principle, with explicit consent as a key exception.""",

        """Conclusion / 結論說明:
        → In conclusion, under the GDPR, consent is one lawful basis for processing personal data, but it is not the only lawful basis.
        → Where processing is based on consent, such consent must be demonstrable and may be withdrawn at any time, without affecting the lawfulness of prior processing.
        → For special categories of personal data, processing is in principle prohibited unless explicit consent or another legal exception applies.

        → 綜合以上條文，在 GDPR 架構下，取得同意僅是處理個人資料的其中一種合法基礎，而非一律必要。
        → 若處理係基於同意，該同意須可被證明，且資料主體得隨時撤回，而不影響撤回前處理之合法性。
        → 至於特殊類別個人資料，原則上禁止處理，除非符合明確同意或其他法定例外。"""

    ],
    "企業於歐盟提供商品或服務所涉之個人資料處理是否適用GDPR?": [
        "Article 3, Paragraph 2 — Territorial scope (Extraterritorial application): "
        "The GDPR applies to the processing of personal data of data subjects who are in the Union, "
        "where the processing activities are related to: "
        "(a) the offering of goods or services, irrespective of whether a payment is required, "
        "to such data subjects in the Union; or "
        "(b) the monitoring of their behaviour as far as their behaviour takes place within the Union. "
        "（GDPR 具有域外適用效力，只要處理行為與向歐盟境內資料主體提供商品或服務，"
        "或對其行為進行監控有關，即適用 GDPR，不論企業是否設立於歐盟境內，也不論是否收費。）",

        "Conclusion / 結論說明: "
        "→ In conclusion, companies that offer goods or services to data subjects in the EU, "
        "or monitor their behaviour within the EU, are subject to the GDPR, regardless of whether payment is required or whether the company is established in the EU. "
        "→ 綜合 Article 3(2) 規定，企業如向歐盟境內資料主體提供商品或服務，"
        "或對其行為進行監控，其相關個人資料處理行為即受 GDPR 規範，"
        "不因是否收取對價或企業是否設立於歐盟而有所不同。"
    ],
    "企業應如何保護個人資料安全?": [
        "Article 5, Paragraph 1(f) — Integrity and confidentiality principle: "
        "Personal data shall be processed in a manner that ensures appropriate security, "
        "including protection against unauthorised or unlawful processing and against accidental loss, destruction or damage, "
        "using appropriate technical or organisational measures. "
        "（個人資料之處理，應以確保適當安全性的方式進行，包括防止未經授權或非法處理，以及防止意外遺失、毀損或損害。）",

        "Interpretative guidance / 解釋指引: "
        "This provision establishes a general principle requiring companies to ensure the confidentiality, "
        "integrity and availability of personal data throughout the processing activities. "
        "Companies are required to adopt appropriate safeguards to prevent unauthorised access, loss or damage, "
        "without prescribing specific technical measures at this level. "
        "（本條為原則性規定，要求企業在整個處理活動中確保個人資料之保密性、完整性與可用性，"
        "並採取適當保護措施，以防止未經授權存取、遺失或毀損。）",

        "Conclusion requirement / 結論輸出要求: "
        "Both the Chinese and English answers must conclude with a clear summary stating that, "
        "under Article 5(1)(f) of the GDPR, companies are required to protect personal data security "
        "by ensuring confidentiality, integrity and availability through appropriate safeguards during processing. "
        "（中英文答案皆須以結論段落總結：依據 GDPR 第 5 條第 1 款第 (f) 項，"
        "企業在處理個人資料時，應透過適當的保護措施，確保資料的保密性、完整性與可用性。）"
     ]


}

# Bilingual question mapping used for parallel Chinese-English answer drafting
# This supports consistency checks across both language versions
zh_to_en = {
    "企業是否能處理特殊類型之個人資料?": "Can companies process special categories of personal data?",
    "資料主體有哪些權利?": "What rights do the data subject have?",
    "個人資料應保存多久?": "How long should personal data be retained?",
    "處理個人資料前是否需要取得同意?": "Is consent required before processing personal data?",
    "企業於歐盟提供商品或服務所涉之個人資料處理是否適用GDPR?": "Is the processing of personal data involving the provision of products or services by companies subject to GDPR?",
    "企業應如何保護個人資料安全?": "How should companies protect the security of personal data?"
}


# =========================
# Bilingual Answer Drafting
# =========================

# Generate draft answers in both Chinese and English
# to enable cross-lingual consistency evaluation during offline QA validation

# Design Note:
# Generating bilingual answers allows consistency checks
# across bilingual legal interpretations in the QA pipeline.

for zh_question, chunks in qa_results.items():
    en_question = zh_to_en[zh_question]

    zh_answer = ask_chatgpt(zh_question, chunks, language="zh")
    en_answer = ask_chatgpt(en_question, chunks, language="en")

    safe_print(f"\n=== Chinese Question: {zh_question} ===")
    safe_print("【中文】\n" + zh_answer)
    safe_print("\n【English】\n" + en_answer)

    safe_print(f"\n=== English question: {en_question} ===")
    safe_print("【中文】\n" + zh_answer)
    safe_print("\n【English】\n" + en_answer)

    safe_print("\n" + "="*80 + "\n")


In [ ]:
# =========================
# Export Initial QA Dataset
# =========================

# Save the drafted QA dataset before manual review
# This serves as an intermediate artifact in the QA refinement workflow
import json

with open("qa_v1.json", "w", encoding="utf-8") as f:
    json.dump(qa_results, f, ensure_ascii=False, indent=4)

print("✅ QA results saved to qa_v1.json")


In [ ]:
# Design Note:
# The finalized QA dataset is embedded only after manual review,
# ensuring that the online retrieval layer is built on verified answers.

import json
from openai import OpenAI
import os

api_key = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# =========================
# Load Finalized QA Dataset
# =========================

# Load the manually reviewed and finalized QA dataset.
# This version is used as the stable answer source for online retrieval.
with open("qa_v1_final.json", "r", encoding="utf-8") as f:
    qa_data = json.load(f)

# Generate embeddings for finalized QA questions
# These embeddings are used during the online retrieval stage
# to match user queries against the validated QA knowledge base
qa_embeddings = {}
for question in qa_data.keys():
    resp = client.embeddings.create(
        model="text-embedding-3-small",
        input=question
    )

    qa_embeddings[question] = resp.data[0].embedding

# Export QA embeddings for downstream online retrieval use
import json
with open("qa_v1_embeddings.json", "w", encoding="utf-8") as f:
    json.dump(qa_embeddings, f, ensure_ascii=False, indent=4)

print("QA embeddings generated successfully.")
